# Fair Lending Lab, hypothesis testing on CFPB HMDA 2023 Massachusetts

End to end reproducible notebook. Every cell takes its random seed from `flab.config.get_random_seed()` so the report is deterministic.

## Outline
1. Connect to the curated `flab.loans` table
2. Summarize denial outcomes by race and income
3. Run the five preregistered hypotheses end to end
4. Apply BH-FDR and Bonferroni across the family
5. Headline narrative and caveats

**Domain caveat:** HMDA omits credit score and full underwriting. Any disparity reported below is a screening signal in observed covariates, not a finding of discrimination.

In [1]:
import json
import pandas as pd
from flab.config import get_random_seed
from flab.db import fetch_df
from flab.hypotheses import REGISTRY, run_hypothesis
from flab.hypotheses.registry import family_wise_correction

print('seed:', get_random_seed())

seed: 20260625


## 1. Row counts

In [2]:
counts = fetch_df(
    "SELECT 'loans' AS t, COUNT(*)::BIGINT AS rows FROM flab.loans "
    "UNION ALL SELECT 'hmda_raw', COUNT(*) FROM flab.hmda_raw"
)
counts

,t,rows
0,loans,41287
1,hmda_raw,210643


## 2. Denial rates by race

In [3]:
by_race = fetch_df(
    "SELECT race_group, COUNT(*)::INT AS n, SUM(denied::int)::INT AS n_denied, "
    "       AVG(denied::int)::FLOAT AS denial_rate "
    "FROM flab.loans GROUP BY race_group ORDER BY n DESC"
)
by_race

,race_group,n,n_denied,denial_rate
0,White,26520,1546,0.058296
1,Not Available,6199,533,0.085982
2,Asian,4974,413,0.083032
3,Black,2124,332,0.156309
4,Joint,1275,73,0.057255
5,Native,89,10,0.112360
6,Multi,63,11,0.174603
7,Pacific,43,8,0.186047


## 3. Run all preregistered hypotheses

In [4]:
rows = []
for k in REGISTRY:
    p = run_hypothesis(k)
    prim = p['primary']
    rows.append({
        'key': k,
        'title': p['title'],
        'method': prim.get('method'),
        'p_value': prim.get('p_value'),
        'effect_size': prim.get('effect_size') or prim.get('eta_squared'),
        'effect_label': prim.get('effect_label'),
        'ci_low': prim.get('ci_low'),
        'ci_high': prim.get('ci_high'),
        'n_a': prim.get('n_a'),
        'n_b': prim.get('n_b'),
    })
summary = pd.DataFrame(rows)
summary

2026-06-26 02:01:44.346 | INFO     | flab.hypotheses.registry:run_hypothesis:527 - running hypothesis


2026-06-26 02:01:44.647 | INFO     | flab.hypotheses.registry:run_hypothesis:527 - running hypothesis


2026-06-26 02:01:44.726 | INFO     | flab.hypotheses.registry:run_hypothesis:527 - running hypothesis


2026-06-26 02:01:48.728 | INFO     | flab.hypotheses.registry:run_hypothesis:527 - running hypothesis


2026-06-26 02:01:48.841 | INFO     | flab.hypotheses.registry:run_hypothesis:527 - running hypothesis


,key,title,method,p_value,effect_size,effect_label,ci_low,ci_high,n_a,n_b
0,h1_denial_race,"Denial rate disparity, Black versus White non-...",two_prop_z,0.000000e+00,0.106574,risk difference,0.089417,0.123731,1806.0,23013.0
1,h2_denial_ethnicity,"Denial rate disparity, Hispanic versus White n...",two_prop_z,0.000000e+00,0.055363,risk difference,0.044172,0.066555,3189.0,23013.0
2,h3_rate_spread_priced,"Rate spread on priced originated loans, Black ...",welch_t,6.314416e-22,0.419803,Hedges' g,0.349686,0.489919,939.0,5000.0
3,h4_lender_effect,Denial rates differ across the top 10 lenders ...,anova_oneway,1.351703e-102,0.031881,NaN,NaN,NaN,NaN,NaN
4,h5_low_income_residual,Denial rate disparity persists within the lowe...,two_prop_z,1.184651e-03,0.185245,risk difference,0.054451,0.316038,60.0,593.0


## 4. Family-wise correction

In [5]:
fwc = family_wise_correction()
pd.DataFrame(fwc['family'])

,hypothesis_key,p_value,p_fdr,reject_fdr,p_bonferroni,reject_bonferroni
0,h1_denial_race,0.000000e+00,0.000000e+00,True,0.000000e+00,True
1,h2_denial_ethnicity,0.000000e+00,0.000000e+00,True,0.000000e+00,True
2,h3_rate_spread_priced,6.314416e-22,7.893020e-22,True,3.157208e-21,True
3,h4_lender_effect,1.351703e-102,2.252838e-102,True,6.758513e-102,True
4,h5_low_income_residual,1.184651e-03,1.184651e-03,True,5.923255e-03,True


## 5. Headline narrative

Black non-Hispanic applicants face a higher denial rate than White non-Hispanic applicants for first-lien conventional owner-occupied home-purchase loans in MA 2023 HMDA. The risk difference is roughly 10.7 percentage points with a 95% CI of [8.9, 12.4], confirmed by the two-proportion z-test (p approx 0), an odds ratio with Haldane-corrected CI, and a BH-FDR adjusted stratified test across income bands. The disparity persists in the lowest income band (H5).

All five primary tests reject the null at BH-FDR q = 0.05 and at Bonferroni.

**Causal caveat.** HMDA omits credit score and detailed underwriting. The result above is consistent with several explanations including residual disparities after partial adjustment, omitted-variable bias from credit-quality factors not present in HMDA, and selection in who applies to which lender. Establishing discrimination requires matched supervisory HMDA data, audit pairs, or randomized testing. This notebook is a screening pass.